# codedet on Colab (manual fallback)

Same code as the Kaggle kernel, same presets, different host. Use this when the Kaggle CLI cannot
push (quota, or a token that only works for read operations).

1. Runtime -> Change runtime type -> **T4 GPU**
2. Runtime -> **Run all**
3. The last cell zips `results/` for download; commit those JSON/MD/PNG files to the repo.

Expect ~5 min for `smoke` and ~40 min for `lite`.

In [ ]:
PRESET = "smoke"  # "smoke" first; switch to "lite" once it passes
REPO = "https://github.com/ranjankndev/codedetection-human-machine.git"
DATASET = "daniilor/SemEval-2026-Task13"  # Hugging Face mirror of the official release

In [ ]:
!git clone -q $REPO codedet
%cd codedet
!pip install -q -e . 2>&1 | tail -2
!pip install -q datasketch lightgbm peft 2>&1 | tail -2
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
# Official data from the Hugging Face mirror (no Kaggle credentials needed)
import os, shutil
from huggingface_hub import hf_hub_download

FILES = ["task_a/task_a_training_set_1.parquet", "task_a/task_a_validation_set.parquet",
         "task_b/task_b_training_set.parquet", "task_b/task_b_validation_set.parquet"]
for f in FILES:
    dest = os.path.join("data", f)
    if os.path.exists(dest):
        continue
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    shutil.copy2(hf_hub_download(DATASET, f, repo_type="dataset"), dest)
    print("downloaded", f)
!python scripts/discover_data.py --data data --out results | tail -2

In [ ]:
# The orchestrator owns the stage DAG, the resume markers and the degradation ladder.
!python -u scripts/run_all.py --preset $PRESET --data data --out results

In [ ]:
import json
print(open("results/summary.md").read()[:4000])
print(json.dumps(json.load(open("results/status.json"))["stages"], indent=1)[:1500])

In [ ]:
# Keep only the small, committable artifacts
!rm -rf results/data results/preds results/ckpt results/cache
!zip -qr codedet_results_$PRESET.zip results && ls -la codedet_results_$PRESET.zip
from google.colab import files; files.download(f"codedet_results_{PRESET}.zip")